Objectif : reduire les données issues de MIRISIM pour faire du RDI (source et reference)

In [1]:
# Packages that allow us to get information about objects:
import asdf
import copy
import os
import shutil
import glob
# Numpy library:
import numpy as np
# For downloading data
import requests
# Astropy tools:
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch
#Set up matplotlib for plotting
import matplotlib.pyplot as plt
import matplotlib as mpl
# parallelisation
#from joblib import Parallel, delayed #besoin d'installer joblib

#solution tempo du notebook MRS
import multiprocessing
multiprocessing.set_start_method('fork')
from multiprocessing import Pool
# Set the maximum number of processes to spawn based on available cores
usage = 'none' # Either 'none' (single thread), 'quarter', 'half', or 'all' available cores
maxp = 1
print('We will use '+str(maxp)+' CPUs in this run')

#Import JWST pipeline-related modules
import jwst
from jwst.pipeline import Detector1Pipeline
from jwst.pipeline import Image2Pipeline
from jwst.pipeline import Coron3Pipeline
from jwst.associations import asn_from_list as afl # Tools for creating association files
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase # Definition of a Lvl2 association file
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base # Definition of a Lvl3 association file
from jwst.associations.lib.member import Member
from jwst.associations.asn_from_list import asn_from_list
from tqdm.notebook import tqdm as tqdm
#Check which version of the pipeline we are running:
print(jwst.__version__)



We will use 1 CPUs in this run
1.12.5


In [2]:
obs_name = 'hr8799_filtreX'
path_code    = '/home/cperrot/Work/code/JWST/Target/GTO/'+obs_name+'/'
path_product = '/home/cperrot/Work/projet/JWST/Target/GTO/'+obs_name+'/'
path_raw     = '/home/cperrot/Work/raw/JWST/Target/GTO/'+obs_name+'/'
path_source  = path_raw+'source/det_images/'
path_ref     = path_raw+'reference/det_images/'

if not os.path.exists(path_product+'source/'):
    os.makedirs(path_product+'source/')
if not os.path.exists(path_product+'reference/'):
    os.makedirs(path_product+'reference/')

#path_raw    = '/home/cperrot/Work/code/MIRISIM/corono/20220328_162102_mirisim/det_images/'

path_stage1 = path_product+'stage1/'
#path_stage2 = '/home/cperrot/Work/code/MIRISIM/corono/corono_S2/'
#path_stage3 = '/home/cperrot/Work/code/MIRISIM/corono/corono_S3/'

if not os.path.exists(path_stage1):
    os.makedirs(path_stage1)
#if not os.path.exists(path_stage2):
#    os.makedirs(path_stage2)
#if not os.path.exists(path_stage3):
#    os.makedirs(path_stage3)

FUNCTION

In [3]:
def rundet1(filenames):
    det1 = Detector1Pipeline() # Instantiate the pipeline
    det1.output_dir = path_stage1 # Specify where the output should go
    det1.refpix.skip = True # Skip the reference pixel subtraction (as it doesn't interact well with simulated data)
    det1.rscd.skip = True
    det1.reset.skip = True
    det1.save_results = True # Save the final resulting _rate.fits files
    det1(filenames) # Run the pipeline on an input list of files
    
# Define a function that can oversee the multiprocessing
# We'll just pass this a function name and a set of inputs, and it'll figure out the
# multithreading accordingly.
def runmany(step, filenames):
    if __name__ == '__main__':
        p = Pool(maxp)
        res = p.map(step, filenames)
        p.close()
        p.join()




def writel2asn(files, asn_name, prodname,**kwargs):
    
    asn = asn_from_list(files, rule=DMSLevel2bBase, product_name=prodname)
    asn.data['products'] = None
    asn['asn_pool'] = asn_name + '_pool'
    asn['asn_type'] = 'image2'
    for n, sci in enumerate(files):
            asn.new_product('{}_exposure{}'.format(asn_name, str(n+1)))
            sci_member = Member({'expname': sci, 'exptype': 'science'})    
            new_members = asn.current_product['members']
            new_members.append(sci_member)
    # print the association and save to file
    name, ser = asn.dump()
    print(ser)
    asn_file2 = path_stage2+asn_name+'.json'
    with open(asn_file2, 'w') as f:
        f.write(ser)
        


In [4]:
# RUN STAGE 1
 
# SOURCE
sstring = path_source+'det*.fits'
raw_files_source = sorted(glob.glob(sstring))
#param_file = path_stage1+'miri_detector1_param_file.asdf'
print("Run stage 1 of the pipeline - SOURCE")
print(f'Found {len(raw_files_source)} input files to process')
#print('Found the param file :'+param_file)
runmany(rundet1, raw_files_source)

# REFERENCE
sstring = path_ref+'det*.fits'
raw_files_ref = sorted(glob.glob(sstring))
#param_file = path_stage1+'miri_detector1_param_file.asdf'
print("Run stage 1 of the pipeline - REFERENCE")
print(f'Found {len(raw_files_ref)} input files to process')
#print('Found the param file :'+param_file)
runmany(rundet1, raw_files_ref)



### faire le déplacement de dossier stage1 -> source_stage_1 et reference_stage_1

Run stage 1 of the pipeline - SOURCE
Found 0 input files to process
Run stage 1 of the pipeline - REFERENCE
Found 0 input files to process


'/home/cperrot/Work/raw/JWST/Simu/pipeline_verification_1/reference/'